In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

df = pd.read_csv("../data/raw/dengue_tulua.csv", low_memory=False)
print(f"Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}")
df.head(3)

## 1. Columns and data types

In [ ]:
info = pd.DataFrame({
    "type": df.dtypes.astype(str),
    "nulls": df.isna().sum(),
    "% nulls": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique(dropna=True),
})
info

### SIVIGILA code dictionary (notification form 210)

Column names are kept exactly as published by SIVIGILA (in Spanish); the meanings are translated here.

| Column | Meaning | Codes |
|---|---|---|
| `tip_cas_` | Case type at notification | 2 = Probable, 3 = Laboratory-confirmed, 5 = Confirmed by epidemiological link |
| `ajuste_` | Later adjustment of the case | 0 = No adjustment, 3 = Laboratory-confirmed, 5 = Confirmed by link, **6 = Discarded**, 7 = Other update |
| `clasfinal` | Clinical classification | 1 = Dengue without warning signs, 2 = Dengue with warning signs, 3 = Severe dengue |
| `pac_hos_` | Hospitalized | 1 = Yes, 2 = No |
| `con_fin_` | Final condition | 1 = Alive, 2 = Dead |
| Symptoms (`fiebre`, `cefalea`, …) | Presence of the sign/symptom | 1 = Yes, 2 = No |
| `famantdngu` | Relative with dengue | 1 = Yes, 2 = No, 3 = Unknown |
| `desplazami` | Travel to an endemic area | 1 = Yes, 2 = No |

**Watch out:** symptoms are coded as **1 = Yes / 2 = No**, not 1/0. They are normalized to 0/1 in Phase 2.

## 2. Null values

In [ ]:
nulls = info[info["nulls"] > 0].sort_values("nulls", ascending=False)
nulls

**Reading:**
- `a_o` (year) has ~77 % nulls; it does not matter, because the year can be taken from `fec_not` (notification date).
- The severe-dengue variables (`extravasac`, `choque`, `da_o_organ`, `mues*`) have 191 nulls. They
  **will not be used** anyway: they are later clinical findings (data leakage).
- The early symptoms **have no nulls**. Even so, the Phase 2 pipeline handles nulls for robustness
  (for example, if a form arrives incomplete).

## 3. Case classification variables

In [ ]:
for col in ["cod_eve", "tip_cas_", "ajuste_", "clasfinal", "con_fin_", "pac_hos_", "conducta"]:
    print(f"--- {col}")
    print(df[col].value_counts(dropna=False).sort_index().to_string(), "\n")

In [ ]:
print("tip_cas_ (rows) vs ajuste_ (columns):")
pd.crosstab(df["tip_cas_"], df["ajuste_"], margins=True)

**Reading:**
- All records are event 210 (dengue) and all patients ended up alive (`con_fin_ = 1`).
- `tip_cas_` is the case type **at notification**. `ajuste_` is the **later correction**, usually when the
  laboratory result arrives. That is why `ajuste_` takes precedence: 30 cases were notified as "confirmed" (3)
  and later discarded (6).
- `clasfinal` only has values 1 and 2 (no severe dengue). It appears even in discarded cases because it is filled in
  at notification, so it **cannot be used as a dengue yes/no label**.

## 4. Are there DISCARDED (non-dengue) cases? → Yes

In [ ]:
df = df.drop_duplicates()
print(f"Rows after removing exact duplicates: {len(df):,}")

discarded = df["ajuste_"] == 6
confirmed = ~discarded & (df["tip_cas_"].isin([3, 5]) | df["ajuste_"].isin([3, 5]))

df["class"] = "Excluded (probable, unconfirmed)"
df.loc[confirmed, "class"] = "Confirmed dengue (1)"
df.loc[discarded, "class"] = "Discarded (0)"

distribution = df["class"].value_counts().to_frame("cases")
distribution["%"] = (distribution["cases"] / distribution["cases"].sum() * 100).round(1)
distribution

**Proposed definition of the target variable (`dengue`)**

| Class | Rule | Reason |
|---|---|---|
| **1 = Dengue** | not discarded **and** (confirmed at notification `tip_cas_ ∈ {3,5}` **or** confirmed by adjustment `ajuste_ ∈ {3,5}`) | The laboratory or the epidemiological link confirm dengue |
| **0 = Not dengue** | `ajuste_ = 6` | The case was investigated and discarded |
| *Excluded* | still "probable" with no confirming adjustment | It was never confirmed nor discarded: including it as 0 or as 1 would add **label noise** |

**Important interpretation for the presentation:** discarded cases are not healthy people. They are patients who
came in with **fever** and compatible symptoms, but whose laboratory result was negative (probably
another viral infection). That is why the model answers:
*"Given a febrile patient with these symptoms, how likely is it to be dengue and not another febrile illness?"*.

In [ ]:
model_df = df[df["class"] != "Excluded (probable, unconfirmed)"].copy()
model_df["dengue"] = (model_df["class"] == "Confirmed dengue (1)").astype(int)
print(model_df["dengue"].value_counts().rename({1: "dengue", 0: "not dengue"}))
print(f"\nShare of the positive class: {model_df['dengue'].mean():.1%}")

ax = model_df["dengue"].map({1: "Dengue", 0: "Not dengue"}).value_counts().plot.bar(color=["#c0392b", "#7f8c8d"], rot=0)
ax.set_title("Class distribution (modeling dataset)")
ax.set_ylabel("Cases")
plt.tight_layout(); plt.show()

**Imbalance:** roughly 70 % dengue / 30 % not dengue. It is a **moderate** imbalance (not extreme).
In Phase 2 it is handled with `class_weight="balanced"` and **stratified** cross-validation, which keeps the
same class proportion in every fold.

## 5. Candidate variables: age and sex

In [ ]:
print("Available age columns:", [c for c in df.columns if "edad" in c or "etario" in c])
print(model_df["grup_etario"].value_counts().sort_index().to_string())
pd.crosstab(model_df["grup_etario"], model_df["dengue"], normalize="index").round(3).rename(columns={0: "% not dengue", 1: "% dengue"})

**Limitation:** this dataset **does not include the exact age** (it was anonymized). It only has ranges. `grup_etario` has
6 consistent ranges (`0-5, 6-11, 12-17, 18-28, 29-59, 60+`). `rango_edad` mixes two different schemes, so
it is discarded. In Phase 2, `grup_etario` is used. The web form asks for the age in years and
converts it to the matching range.

In [ ]:
pd.crosstab(model_df["sexo_"], model_df["dengue"], normalize="index").round(3).rename(columns={0: "% not dengue", 1: "% dengue"})

## 6. Symptoms: which ones separate dengue from not dengue?

In [ ]:
SYMPTOMS = {
    "fiebre": "Fever", "cefalea": "Headache", "dolrretroo": "Pain behind the eyes",
    "malgias": "Muscle pain", "artralgia": "Joint pain", "erupcionr": "Skin rash",
    "dolor_abdo": "Abdominal pain", "vomito": "Vomiting", "diarrea": "Diarrhea",
    "somnolenci": "Drowsiness", "hipotensio": "Hypotension", "hepatomeg": "Enlarged liver",
    "hem_mucosa": "Mucosal bleeding", "hipotermia": "Hypothermia", "aum_hemato": "Increased hematocrit",
    "caida_plaq": "Platelet drop", "acum_liqui": "Fluid accumulation",
    "famantdngu": "Relative with dengue", "desplazami": "Traveled to endemic area",
}
prevalence = pd.DataFrame({
    "% in dengue": (model_df.loc[model_df.dengue == 1, list(SYMPTOMS)] == 1).mean() * 100,
    "% in not dengue": (model_df.loc[model_df.dengue == 0, list(SYMPTOMS)] == 1).mean() * 100,
})
prevalence["difference (pp)"] = prevalence["% in dengue"] - prevalence["% in not dengue"]
prevalence.index = [f"{SYMPTOMS[c]} ({c})" for c in prevalence.index]
prevalence.round(1).sort_values("difference (pp)", ascending=False)

In [ ]:
p = prevalence.sort_values("difference (pp)")
ax = p[["% in not dengue", "% in dengue"]].plot.barh(figsize=(8, 7), color=["#7f8c8d", "#c0392b"])
ax.set_xlabel("% of patients with the symptom"); ax.set_title("Symptom prevalence by class")
plt.tight_layout(); plt.show()

**Reading (important for the model's expectations):**
- **Fever is present in 100 % of cases.** It is an entry criterion to notify dengue, so it **adds no
  information** and is excluded as a variable.
- The largest difference is **platelet drop** (~26 % vs ~8 %). But it is a **laboratory result**
  (blood count), not a symptom the patient can report. See the pending decision below.
- Among reportable symptoms, the ones that separate classes the most are **rash**, **abdominal pain**, **headache**,
  **retro-orbital pain** and **vomiting**, with differences of 4 to 10 percentage points.
- Muscle pain, joint pain and diarrhea are **almost equal** in both classes: other febrile viral infections
  cause them too.
- **Honest conclusion:** with reportable symptoms only, the signal is **moderate**. Expect a ROC-AUC of
  roughly 0.60 to 0.70, not 0.90. That is consistent with medicine: dengue is confirmed with laboratory tests
  precisely because symptoms alone are not enough.

## 7. Time distribution and date quality

In [ ]:
model_df["notification_year"] = pd.to_datetime(model_df["fec_not"]).dt.year
t = pd.crosstab(model_df["notification_year"], model_df["dengue"]).rename(columns={0: "not dengue", 1: "dengue"})
t["% dengue"] = (t["dengue"] / t.sum(axis=1) * 100).round(1)
t

In [ ]:
days = (pd.to_datetime(df["fec_not"]) - pd.to_datetime(df["ini_sin_"])).dt.days
days.describe()

**Reading:**
- Most cases are from **2024** (an epidemic year in Colombia).
- In **2025–2026** the share of confirmed dengue drops a lot. There may be more cases still "under study"
  or a change in testing policy. We note it as a **limitation**: the class mix changes over time.
- The median time between symptom onset and notification is 5 days.

## 8. Variable classification for Phase 2

| Group | Variables | Decision |
|---|---|---|
| **Input (form)** | `grup_etario`, `sexo_`, `cefalea`, `dolrretroo`, `malgias`, `artralgia`, `erupcionr`, `dolor_abdo`, `vomito`, `diarrea`, `somnolenci`, `hem_mucosa` | Use |
| Reportable context (optional) | `famantdngu`, `desplazami` | To be decided (almost no difference between classes) |
| No information | `fiebre` | Exclude (constant = 1) |
| **Laboratory / medical exam** | `caida_plaq`, `aum_hemato`, `hipotensio`, `hepatomeg`, `acum_liqui`, `hipotermia` | **Pending decision** (see below) |
| **Data leakage** | `choque`, `da_o_organ`, `extravasac`, `hemorr_hem`, `clasfinal`, `conducta`, `pac_hos_`, `con_fin_`, `tip_cas_`, `ajuste_`, `mues*` | Always exclude |
| Administrative | `fec_not`, `semana`, `cod_ase_`, `estrato_`, `gp_*`, `nmun_resi`, etc. | Exclude |

### Pending decision
Laboratory or medical-exam variables (especially `caida_plaq`) would help the most, but a
patient **cannot report them** in a form. The options are:
- **A (recommended, consistent with the assignment):** reportable symptoms only. The model is weaker, but honest.
- **B:** add `caida_plaq` as an optional field "Did you have a blood count that showed low platelets?".
  It improves the model, but changes the form's audience to people who already had exams.